# Support Vector Machines — Exercises

Companion to the note [Support Vector Machines](Support%20Vector%20Machines.md).

Practise the geometry of the margin, hard- and soft-margin SVMs, hinge loss, the Lagrangian dual and KKT conditions, support vectors, kernels and the tuning of $C$ and $\gamma$, by hand and from scratch with NumPy/SciPy (compared against `sklearn.svm`).

**17 exercises** · 🧠 Concept ×4 · ✍️ By hand ×7 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.optimize import minimize
from sklearn.svm import SVC, SVR
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons
rng = np.random.default_rng(0)
# Two overlapping Gaussian blobs, labels in {-1, +1}
Xs = np.vstack([rng.normal(size=(100, 2)) + [1.5, 1.5], rng.normal(size=(100, 2)) - [1.5, 1.5]])
ys = np.r_[np.ones(100), -np.ones(100)]

## Part A · Concepts

### Exercise 1 · Why the maximum margin?
*🧠 Concept · ★☆☆*

Many hyperplanes separate a separable dataset (the [[Linear Models for Classification|perceptron]] finds just one of them). Give two arguments for choosing the
one with the **maximum margin**, and explain what a *support vector* is.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think of robustness to small perturbations of the points, and of what happens to the solution when you delete a non-support point.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(1) **Robustness**: every training point can move by up to half the margin without being misclassified, so small noise in test points is tolerated.
(2) **Generalisation**: margin-based bounds depend on $R^2/\gamma^2$ rather than the dimension (compare the perceptron bound), and the solution is determined by few points:
leave-one-out error $\le \#\text{SV}/N$. **Support vectors** are the points on or inside the margin ($\alpha_i>0$); deleting any other point leaves the solution unchanged.

</details>

### Exercise 2 · Reading $C$ and $\gamma$
*🧠 Concept · ★★☆*

For an RBF SVM, describe the decision boundary and the risk of under-/overfitting for: (a) small $C$, small $\gamma$; (b) large $C$, large $\gamma$;
(c) large $C$, small $\gamma$. Why must the features be scaled first?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$\gamma$ is an inverse squared length scale: each support vector influences a region of radius about $1/\sqrt\gamma$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Wide margin, many margin violations allowed and each SV influences a large region: very smooth, nearly linear boundary → **underfitting** (high bias).
(b) Few violations tolerated and very local kernels: boundary wraps around individual points, almost every point is an SV → **overfitting**.
(c) Smooth kernel but hard penalty: fairly smooth, can still fit well; this corner often works with properly tuned values.
Scaling matters because $\lVert x-x'\rVert^2$ mixes all features: a feature in large units dominates the kernel (same issue as in [[k-Nearest Neighbors]]).
Tune $C,\gamma$ on a log grid (Exercise 15).

</details>

### Exercise 3 · SVM vs logistic regression
*🧠 Concept · ★☆☆*

Both are linear classifiers trained with an L2 penalty. Compare the hinge loss $\max(0,1-yf)$ and the logistic loss $\log(1+e^{-yf})$:
which points contribute to each solution, and which gives probabilities? What is Platt scaling?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Sketch both losses as functions of the margin $yf$. Where is each exactly zero?

</details>

<details>
<summary><b>✅ Solution</b></summary>

The hinge loss is **exactly 0** for $yf\ge1$, so well-classified points beyond the margin have no influence: the solution is **sparse** (only support vectors).
The logistic loss is positive everywhere, so every point contributes, with weight decaying as $yf$ grows. Logistic regression outputs calibrated probabilities
([[Logistic Regression]]); the SVM outputs an uncalibrated score $f(x)$. **Platt scaling** fits $p(y=1\mid x)=\sigma(af(x)+b)$ on held-out scores
(`SVC(probability=True)` does this with internal CV).

</details>

### Exercise 4 · Primal or dual?
*🧠 Concept · ★★☆*

The primal has $d+1$ variables, the dual $N$. When would you solve each? Why does only the dual allow the kernel trick, and what does it cost in memory and time?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Look at how the data enter the dual objective.

</details>

<details>
<summary><b>✅ Solution</b></summary>

**Primal** (e.g. `LinearSVC`, SGD): good for linear SVMs with huge $N$ and moderate $d$ (or sparse text features); cost roughly linear in $N$.
**Dual**: data enter only via dot products $x_i^\top x_j$, which can be replaced by any kernel $k(x_i,x_j)$ ([[Kernel Methods]]), giving non-linear boundaries
without ever computing $\phi(x)$. The price: the $N\times N$ Gram matrix ($O(N^2)$ memory) and $O(N^2)$–$O(N^3)$ training time, so kernel SVMs struggle beyond ~$10^5$ points.

</details>

## Part B · By hand

### Exercise 5 · Margins and distances
*✍️ By hand · ★☆☆*

A canonical SVM hyperplane has $w=(3,4)$, $b=-5$. (a) What is the margin width $2/\lVert w\rVert$? (b) For $x=(2,1)$, $y=+1$, compute the functional margin
$y(w^\top x+b)$ and the geometric distance to the hyperplane. Is $x$ on, inside or outside the margin?

In [ ]:
margin_width = None
func_margin = None
geo_dist = None

_w = np.array([3, 4.]); _b = -5; _x = np.array([2, 1.])
check('margin width', margin_width, 2/np.linalg.norm(_w)); check('functional margin', func_margin, _w@_x+_b)
check('geometric distance', geo_dist, (_w@_x+_b)/np.linalg.norm(_w))

<details>
<summary><b>💡 Hint</b></summary>

$\lVert w\rVert=5$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $2/5=0.4$. (b) Functional margin $6+4-5=5$; distance $5/5=1$. Since $y f(x)=5\ge1$ the point is well **outside** the margin (each side of the margin is $1/\lVert w\rVert=0.2$ wide).

```python
margin_width = 0.4
func_margin = 5
geo_dist = 1.0
```

</details>

### Exercise 6 · A hard-margin SVM with two points
*✍️ By hand · ★☆☆*

Training set: $x_+=(2,2)$ with $y=+1$ and $x_-=(0,0)$ with $y=-1$. Find the hard-margin SVM $(w,b)$ and the margin width.

In [ ]:
w_two = None
b_two = None
margin_two = None

_m = SVC(kernel='linear', C=1e6).fit(np.array([[2, 2], [0, 0.]]), [1, -1])
check('w', w_two, _m.coef_[0], tol=1e-3); check('b', b_two, _m.intercept_[0], tol=1e-3)
check('margin width', margin_two, 2/np.linalg.norm(_m.coef_[0]), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

By symmetry $w\propto x_+-x_-$, and both points are support vectors: $w^\top x_++b=1$, $w^\top x_-+b=-1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$w=c(1,1)$. From $b=-1$ (the point $x_-=0$) and $2c+2c+b=1$: $c=\tfrac12$. So $w=(0.5,0.5)$, $b=-1$, margin $2/\lVert w\rVert=2/0.7071=2\sqrt2\approx2.828$,
exactly the distance between the two points: the boundary is their perpendicular bisector $x_1+x_2=2$.

```python
w_two = [0.5, 0.5]
b_two = -1.0
margin_two = 2 * np.sqrt(2)
```

</details>

### Exercise 7 · Hinge loss and the soft-margin objective
*✍️ By hand · ★☆☆*

Four points have $y=+1$ and scores $f(x)=(2,\ 1,\ 0.5,\ -1)$. (a) Compute each hinge loss $\max(0,1-yf)$, i.e. the slacks $\xi_i$.
(b) With $w=(1,1)$ and $C=0.5$, evaluate the soft-margin objective $\tfrac12\lVert w\rVert^2+C\sum\xi_i$.

In [ ]:
xi = None   # 4 numbers
obj_soft = None

_f = np.array([2, 1, .5, -1.])
check('slacks', xi, np.maximum(0, 1-_f))
check('objective', obj_soft, 0.5*2 + 0.5*np.maximum(0, 1-_f).sum())

<details>
<summary><b>💡 Hint</b></summary>

Point 2 sits exactly on the margin; point 3 is inside the margin but correctly classified; point 4 is misclassified.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\xi=(0,\ 0,\ 0.5,\ 2)$. Objective $=\tfrac12\cdot2+0.5\cdot2.5=2.25$. A slack $0<\xi<1$ means 'inside the margin but correct'; $\xi>1$ means misclassified, so $\sum\xi_i$
upper-bounds the number of training errors.

```python
xi = [0, 0, 0.5, 2]
obj_soft = 2.25
```

</details>

### Exercise 8 · Soft margin = hinge loss + L2
*✍️ By hand · ★★☆*

Show that the constrained soft-margin problem of the note is equivalent to the unconstrained problem
$\min_{w,b}\ \tfrac12\lVert w\rVert^2+C\sum_i\max(0,1-y_i(w^\top x_i+b))$. Rewrite it as 'loss + regulariser' with $\lambda=1/(CN)$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

For fixed $(w,b)$, what is the smallest $\xi_i$ that satisfies both $\xi_i\ge0$ and $\xi_i\ge1-y_if(x_i)$?

</details>

<details>
<summary><b>✅ Solution</b></summary>

For fixed $(w,b)$ the objective increases in each $\xi_i$, so the optimal slack is the smallest feasible one: $\xi_i=\max(0,1-y_if(x_i))$. Substituting removes the constraints.
Dividing by $CN$: $\frac1N\sum_i\max(0,1-y_if(x_i))+\frac{\lambda}{2}\lVert w\rVert^2$ with $\lambda=\frac1{CN}$, i.e. empirical hinge risk + L2
([[Overfitting and Regularization]]). Large $C$ = weak regularisation.

</details>

### Exercise 9 · Deriving the dual
*✍️ By hand · ★★★*

Form the Lagrangian of the soft-margin primal with multipliers $\alpha_i\ge0$ (for $y_i(w^\top x_i+b)\ge1-\xi_i$) and $\mu_i\ge0$ (for $\xi_i\ge0$).
Set the derivatives with respect to $w,b,\xi$ to zero and derive the dual of the note, including the box constraint $0\le\alpha_i\le C$ and $\sum_i\alpha_iy_i=0$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$\mathcal L=\tfrac12\lVert w\rVert^2+C\sum\xi_i-\sum\alpha_i[y_i(w^\top x_i+b)-1+\xi_i]-\sum\mu_i\xi_i$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\partial_w\mathcal L=0\Rightarrow w=\sum_i\alpha_iy_ix_i$. $\partial_b\mathcal L=0\Rightarrow\sum_i\alpha_iy_i=0$. $\partial_{\xi_i}\mathcal L=0\Rightarrow C-\alpha_i-\mu_i=0$, and with $\mu_i\ge0$: $\alpha_i\le C$.
Substituting, the $\xi$ terms cancel and $b$ drops out:
$\mathcal L=\tfrac12\sum_{ij}\alpha_i\alpha_jy_iy_jx_i^\top x_j-\sum_{ij}\alpha_i\alpha_jy_iy_jx_i^\top x_j+\sum_i\alpha_i
=\sum_i\alpha_i-\tfrac12\sum_{ij}\alpha_i\alpha_jy_iy_jx_i^\top x_j$, maximised over $0\le\alpha_i\le C$, $\sum\alpha_iy_i=0$.
Since the primal is convex with affine constraints, strong duality holds ([[Calculus and Optimization]]). You will solve this QP numerically in Exercise 13.

</details>

### Exercise 10 · Reading the KKT conditions
*✍️ By hand · ★★☆*

Complementary slackness gives: $\alpha_i=0\Rightarrow y_if(x_i)\ge1$; $0<\alpha_i<C\Rightarrow y_if(x_i)=1$; $\alpha_i=C\Rightarrow y_if(x_i)\le1$.
With $C=1$ and $\alpha=(0,\ 0.3,\ 1,\ 0,\ 1,\ 0.7)$ for points $0..5$: list the indices of (a) the support vectors, (b) the points exactly on the margin,
(c) the points that may violate the margin.

In [ ]:
sv_idx = None       # list of indices
on_margin = None
may_violate = None

_a = np.array([0, .3, 1, 0, 1, .7]); _C = 1
check('support vectors', sv_idx, list(np.where(_a > 0)[0]))
check('on the margin', on_margin, list(np.where((_a > 0) & (_a < _C))[0]))
check('possibly inside/violating the margin', may_violate, list(np.where(_a == _C)[0]))

<details>
<summary><b>💡 Hint</b></summary>

Support vectors are exactly the points with $\alpha_i>0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) SVs: $\{1,2,4,5\}$. (b) Free SVs on the margin ($0<\alpha<C$): $\{1,5\}$; these are used to compute $b$ (Exercise 11).
(c) Bounded SVs ($\alpha=C$): $\{2,4\}$, inside the margin or misclassified. Points 0 and 3 lie outside the margin and do not affect $w$.

```python
sv_idx = [1, 2, 4, 5]
on_margin = [1, 5]
may_violate = [2, 4]
```

</details>

### Exercise 11 · Recovering $w$ and $b$ from the dual
*✍️ By hand · ★★☆*

A hard-margin dual solution has two support vectors: $x_1=(2,0.5)$, $y_1=+1$ and $x_2=(0.5,2)$, $y_2=-1$, with $\alpha_1=\alpha_2=\tfrac49$.
Check $\sum\alpha_iy_i=0$, compute $w=\sum\alpha_iy_ix_i$ and $b$ from the margin condition $y_1(w^\top x_1+b)=1$, and verify that $x_2$ is on the margin too.

In [ ]:
w_dual = None
b_dual = None

_m = SVC(kernel='linear', C=1e6).fit(np.array([[2, .5], [.5, 2]]), [1, -1])
check('w', w_dual, _m.coef_[0], tol=1e-3); check('b', b_dual, _m.intercept_[0], tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$w=\tfrac49\big[(2,0.5)-(0.5,2)\big]$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\sum\alpha_iy_i=\tfrac49-\tfrac49=0$ ✔. $w=\tfrac49(1.5,-1.5)=(\tfrac23,-\tfrac23)$. $w^\top x_1=\tfrac23(2-0.5)=1$, so $b=1-1=0$.
Check: $w^\top x_2+b=\tfrac23(0.5-2)=-1$, so $y_2f(x_2)=1$ ✔: both points lie exactly on the margin. In general, for two points
$\alpha=2/\lVert x_1-x_2\rVert^2$ (here $2/4.5$). Only the support vectors and their $\alpha$s are needed to make predictions.

```python
w_dual = [2/3, -2/3]
b_dual = 0.0
```

</details>

## Part C · Code from scratch

### Exercise 12 · Primal linear SVM by subgradient descent
*💻 Code · ★★☆*

Minimise $J(w,b)=\tfrac12\lVert w\rVert^2+C\sum_i\max(0,1-y_i(w^\top x_i+b))$ with full-batch subgradient descent on `Xs, ys` ($C=1$),
step size $\eta_t=\frac{1}{50\sqrt t}$, 20 000 iterations, keeping the best iterate. Compare your objective with sklearn's `SVC(kernel='linear', C=1)`.

In [ ]:
def svm_objective(w, b, X, y, C=1.0):
    # TODO
    return None

def svm_subgradient(X, y, C=1.0, iters=20000):
    # TODO: return best (w, b)
    return None

wb_sg = svm_subgradient(Xs, ys)

_m = SVC(kernel='linear', C=1.0).fit(Xs, ys)
_obj = 0.5*_m.coef_[0]@_m.coef_[0] + np.maximum(0, 1 - ys*(Xs@_m.coef_[0] + _m.intercept_[0])).sum()
check('objective function', svm_objective(_m.coef_[0], _m.intercept_[0], Xs, ys), _obj)
check('subgradient reaches sklearn objective (within 0.5%)', None if wb_sg is None else svm_objective(*wb_sg, Xs, ys) <= 1.005*_obj, True)

<details>
<summary><b>💡 Hint</b></summary>

A subgradient: $w-C\sum_{i:\,y_if_i<1}y_ix_i$ for $w$ and $-C\sum_{i:\,y_if_i<1}y_i$ for $b$. The objective is not monotone under subgradient steps, hence 'keep the best'.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The hinge is not differentiable at $yf=1$, so we use a subgradient and a decreasing step size; convergence is slow ($O(1/\sqrt t)$) but reaches the optimum to ~$10^{-5}$ here.
Pegasos (stochastic version with $\eta_t=1/(\lambda t)$) is the classic large-scale variant.

```python
def svm_objective(w, b, X, y, C=1.0):
    return 0.5 * w @ w + C * np.maximum(0, 1 - y * (X @ w + b)).sum()

def svm_subgradient(X, y, C=1.0, iters=20000):
    w, b = np.zeros(X.shape[1]), 0.0
    best, best_wb = np.inf, (w.copy(), b)
    for t in range(1, iters + 1):
        act = y * (X @ w + b) < 1
        gw = w - C * (y[act, None] * X[act]).sum(0)
        gb = -C * y[act].sum()
        lr = 1 / (50 * np.sqrt(t))
        w, b = w - lr * gw, b - lr * gb
        o = svm_objective(w, b, X, y, C)
        if o < best:
            best, best_wb = o, (w.copy(), b)
    return best_wb

wb_sg = svm_subgradient(Xs, ys)
print('w, b =', wb_sg)
```

</details>

### Exercise 13 · Solving the dual QP
*💻 Code · ★★★*

Solve the dual of the note for the first 20 and last 20 points of `Xs, ys` ($C=1$) with `scipy.optimize.minimize(method='SLSQP')`
(bounds $0\le\alpha_i\le C$, equality $\sum\alpha_iy_i=0$). Recover $w$, compute $b$ by averaging $y_i-w^\top x_i$ over free support vectors, and compare with sklearn.

In [ ]:
Xd = np.r_[Xs[:20], Xs[-20:]]; yd = np.r_[ys[:20], ys[-20:]]
def svm_dual(X, y, C=1.0):
    # TODO: return (alpha, w, b)
    return None

dual_sol = svm_dual(Xd, yd)

_m = SVC(kernel='linear', C=1.0).fit(Xd, yd)
if dual_sol is None:
    check('w from the dual', None, 0); check('b', None, 0)
else:
    check('w from the dual', dual_sol[1], _m.coef_[0], tol=1e-3)
    check('b', dual_sol[2], _m.intercept_[0], tol=1e-2)

<details>
<summary><b>💡 Hint 1</b></summary>

Minimise the negative dual: `0.5 * a @ Q @ a - a.sum()` with `Q = (y[:,None]*y[None,:]) * (X @ X.T)`; its gradient is `Q @ a - 1`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Free SVs: `(a > 1e-5) & (a < C - 1e-5)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

SLSQP handles the box and the single equality constraint; real solvers use SMO (optimise two $\alpha$s at a time analytically), as libsvm inside sklearn does.
Only a handful of $\alpha_i$ are non-zero: those are the support vectors.

```python
Xd = np.r_[Xs[:20], Xs[-20:]]; yd = np.r_[ys[:20], ys[-20:]]
def svm_dual(X, y, C=1.0):
    Q = (y[:, None] * y[None, :]) * (X @ X.T)
    res = minimize(lambda a: 0.5 * a @ Q @ a - a.sum(), np.zeros(len(y)), jac=lambda a: Q @ a - 1,
                   bounds=[(0, C)] * len(y), constraints=[{'type': 'eq', 'fun': lambda a: a @ y, 'jac': lambda a: y}],
                   method='SLSQP', options={'ftol': 1e-12, 'maxiter': 1000})
    a = res.x
    w = (a * y) @ X
    free = (a > 1e-5) & (a < C - 1e-5)
    b = np.mean(y[free] - X[free] @ w)
    return a, w, b

dual_sol = svm_dual(Xd, yd)
print('support vectors:', np.sum(dual_sol[0] > 1e-5), '| w =', dual_sol[1], '| b =', dual_sol[2])
```

</details>

### Exercise 14 · Kernel SVM predictions by hand
*💻 Code · ★★☆*

Fit `SVC(kernel='rbf', gamma=0.5, C=1)` on `make_moons(200, noise=0.2, random_state=0)`. Using only `dual_coef_` ($=\alpha_iy_i$), `support_vectors_` and `intercept_`,
compute $f(x)=\sum_{i\in SV}\alpha_iy_i\,k(x_i,x)+b$ yourself for a grid of points and compare with `decision_function`.

In [ ]:
Xm, ym = make_moons(200, noise=0.2, random_state=0)
svc = SVC(kernel='rbf', gamma=0.5, C=1.0).fit(Xm, ym)
grid = np.c_[np.linspace(-1.5, 2.5, 30), np.linspace(-1, 1.5, 30)]
f_manual = None

check('manual decision function', f_manual, svc.decision_function(grid))

<details>
<summary><b>💡 Hint</b></summary>

`K = np.exp(-gamma * sqdist(grid, SV))` then `K @ dual_coef_[0] + intercept_[0]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Prediction only needs the support vectors (here a minority of the 200 points), never $\phi(x)$, which for the RBF kernel is infinite-dimensional.
Prediction cost is $O(\#SV\cdot d)$ per point.

```python
Xm, ym = make_moons(200, noise=0.2, random_state=0)
svc = SVC(kernel='rbf', gamma=0.5, C=1.0).fit(Xm, ym)
grid = np.c_[np.linspace(-1.5, 2.5, 30), np.linspace(-1, 1.5, 30)]
SV = svc.support_vectors_
D = (grid ** 2).sum(1)[:, None] + (SV ** 2).sum(1)[None, :] - 2 * grid @ SV.T
f_manual = np.exp(-0.5 * D) @ svc.dual_coef_[0] + svc.intercept_[0]
print('number of SVs:', len(SV))
```

</details>

### Exercise 15 · Tuning $C$ and $\gamma$ on a log grid
*💻 Code · ★★☆*

On `make_moons(300, noise=0.3, random_state=1)`, grid-search $C\in\{10^{-2},\dots,10^{3}\}$ and $\gamma\in\{10^{-2},\dots,10^{2}\}$ (powers of 10) with 5-fold CV over a
`make_pipeline(StandardScaler(), SVC())`. Store the best parameters and the best CV accuracy. Print the CV accuracy table and identify the under- and overfitting corners.

In [ ]:
Xg, yg = make_moons(300, noise=0.3, random_state=1)
Cs = 10.0 ** np.arange(-2, 4); gammas = 10.0 ** np.arange(-2, 3)
best_C = None
best_gamma = None
best_cv = None

_gs = GridSearchCV(make_pipeline(StandardScaler(), SVC()), {'svc__C': Cs, 'svc__gamma': gammas}, cv=5).fit(Xg, yg)
check('best C', best_C, _gs.best_params_['svc__C']); check('best gamma', best_gamma, _gs.best_params_['svc__gamma'])
check('best CV accuracy', best_cv, _gs.best_score_)

<details>
<summary><b>💡 Hint</b></summary>

`GridSearchCV(pipe, {'svc__C': Cs, 'svc__gamma': gammas}, cv=5)`; reshape `cv_results_['mean_test_score']` to (len(Cs), len(gammas)).

</details>

<details>
<summary><b>✅ Solution</b></summary>

With small $\gamma$ (left columns) the boundary is nearly linear and accuracy stays around 0.83–0.84 whatever $C$ is (underfitting);
with $\gamma=100$ (right column) each SV only influences a tiny neighbourhood and accuracy drops (0.76–0.83, overfitting).
The good settings form a diagonal band: larger $C$ pairs with smaller $\gamma$ (best here ≈ 0.91 at $C=1000$, $\gamma=0.1$; $C=1$, $\gamma=1$ is almost as good).
Searching on a log scale matters because both parameters act multiplicatively.

```python
Xg, yg = make_moons(300, noise=0.3, random_state=1)
Cs = 10.0 ** np.arange(-2, 4); gammas = 10.0 ** np.arange(-2, 3)
gs = GridSearchCV(make_pipeline(StandardScaler(), SVC()), {'svc__C': Cs, 'svc__gamma': gammas}, cv=5).fit(Xg, yg)
table = gs.cv_results_['mean_test_score'].reshape(len(Cs), len(gammas))
print('rows: C =', Cs, '\ncols: gamma =', gammas)
print(np.round(table, 3))
best_C, best_gamma, best_cv = gs.best_params_['svc__C'], gs.best_params_['svc__gamma'], gs.best_score_
```

</details>

### Exercise 16 · Support vectors vs $C$
*💻 Code · ★☆☆*

Fit `SVC(kernel='linear', C=C)` on `Xs, ys` for $C\in\{0.001, 0.01, 0.1, 1, 10, 100\}$ and record the number of support vectors (`n_sv`) and the margin
width $2/\lVert w\rVert$ (`widths`). Explain the trend.

In [ ]:
C_list = [0.001, 0.01, 0.1, 1, 10, 100]
n_sv = None
widths = None

check('number of SVs decreases with C', None if n_sv is None else bool(np.all(np.diff(n_sv) <= 0)) and n_sv[0] > 3 * n_sv[-1], True)
check('margin shrinks with C', None if widths is None else bool(np.all(np.diff(widths) <= 1e-3)), True)

<details>
<summary><b>💡 Hint</b></summary>

`len(model.support_)` and `model.coef_`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Small $C$ makes slack cheap: a wide margin containing most of the points, so almost all points are (bounded) support vectors. Large $C$ penalises violations,
so the margin shrinks and only the points near the overlap region remain SVs. This is the bias–variance knob of the note: small $C$ → wide margin, large $C$ → low bias.

```python
C_list = [0.001, 0.01, 0.1, 1, 10, 100]
n_sv, widths = [], []
for C in C_list:
    m = SVC(kernel='linear', C=C).fit(Xs, ys)
    n_sv.append(len(m.support_)); widths.append(2 / np.linalg.norm(m.coef_[0]))
print(n_sv); print(np.round(widths, 3))
```

</details>

### Exercise 17 · SVR and the ε-insensitive tube
*💻 Code · ★★☆*

Support vector regression ignores errors smaller than $\varepsilon$: loss $\max(0,|y-f(x)|-\varepsilon)$. Implement this loss as `eps_insensitive(r, eps)`. Then fit
`SVR(kernel='rbf', C=10, epsilon=eps)` to noisy sine data for $\varepsilon\in\{0.01, 0.1, 0.3, 0.6\}$ and record the number of support vectors.
Check that the points strictly inside the tube are exactly the non-support vectors.

In [ ]:
xr = np.sort(rng.uniform(0, 6, 120))[:, None]
yr = np.sin(xr[:, 0]) + 0.2 * rng.normal(size=120)
def eps_insensitive(r, eps):
    # TODO
    return None
eps_list = [0.01, 0.1, 0.3, 0.6]
n_sv_svr = None

check('loss', eps_insensitive(np.array([-1, -.05, 0, .2, .5]), 0.1), np.maximum(0, np.abs([-1, -.05, 0, .2, .5]) - 0.1))
check('fewer SVs for a wider tube', None if n_sv_svr is None else bool(np.all(np.diff(n_sv_svr) < 0)), True)
_m = SVR(kernel='rbf', C=10, epsilon=0.3).fit(xr, yr)
_inside = np.abs(yr - _m.predict(xr)) < 0.3 - 1e-3
check('inside the tube <=> not a support vector', None if n_sv_svr is None else not np.any(np.isin(np.where(_inside)[0], _m.support_)), True)

<details>
<summary><b>💡 Hint</b></summary>

Points with $|y_i-f(x_i)|<\varepsilon$ have zero loss and zero dual variables, just like points outside the margin in classification.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Widening the tube makes more points fit 'for free', so the number of SVs falls sharply (from nearly all points at $\varepsilon=0.01$ to a handful at $0.6$). The same sparsity
mechanism as the hinge loss: only points on or outside the tube carry non-zero $\alpha$.

```python
xr = np.sort(rng.uniform(0, 6, 120))[:, None]
yr = np.sin(xr[:, 0]) + 0.2 * rng.normal(size=120)
def eps_insensitive(r, eps):
    return np.maximum(0, np.abs(r) - eps)
eps_list = [0.01, 0.1, 0.3, 0.6]
n_sv_svr = [len(SVR(kernel='rbf', C=10, epsilon=e).fit(xr, yr).support_) for e in eps_list]
print(n_sv_svr)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Support Vector Machines](Support%20Vector%20Machines.md).*